# Cross-Framework Comparison - Logistic Regression

Global imports

In [ ]:
from river import linear_model, optim
from tabulate import tabulate
import os
import importlib.util
import sys

# We use an external library (scikit-learn) to compute the metrics consistently across models
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

Initial environment configuration

In [ ]:
# Ensure custom capymoa wrapper is loaded over the installed one
wrapper_path = os.path.abspath(os.path.join(os.getcwd(), "..", "capymoa", "src"))
if wrapper_path not in sys.path:
    sys.path.insert(0, wrapper_path)

os.environ["CAPYMOA_MOA_JAR"] = os.path.abspath(os.path.join(os.getcwd(), "..", "custom_moa_full.jar"))

Global variables

In [ ]:
DEFAULT_LR = 0.01
DEFAULT_BIAS_LR = 0.01
DEFAULT_L1 = 0.0
DEFAULT_L2 = 0.0
DEFAULT_CLIP = 1e12
DEFAULT_BIAS_INIT = 0.0
MAX_INSTANCES = 100000
SEED = 42

Dynamically load the custom LogisticRegression over the installed capymoa package

In [ ]:
file_path = os.path.abspath(os.path.join(os.getcwd(), "..", "capymoa", "src", "capymoa", "classifier", "_logistic_regression.py"))
spec = importlib.util.spec_from_file_location("capymoa.classifier._logistic_regression", file_path)
module = importlib.util.module_from_spec(spec)
sys.modules["capymoa.classifier._logistic_regression"] = module
spec.loader.exec_module(module)
LogisticRegression = module.LogisticRegression

Global functions

In [ ]:
def adaptStreamForRiver(stream):
    data = []

    for i, instance in enumerate(stream):
        if(i > MAX_INSTANCES): break

        # features
        x = {f"f{j}": float(v) for j, v in enumerate(instance.x)}

        # label
        y = instance.y_index
        
        data.append((x, y))
    
    return data

In [ ]:
def evaluateStream(stream_factory, lr=DEFAULT_LR, b_lr=DEFAULT_BIAS_LR, l1=DEFAULT_L1, l2=DEFAULT_L2, clip=DEFAULT_CLIP, bias_init=DEFAULT_BIAS_INIT):
    # stream_factory must be deterministic. It is the constructor of the stream.

    capyMoaResults = _evaluateStreamOnCapyMoa(stream_factory(), lr, b_lr, l1, l2, clip, bias_init)

    riverStream = adaptStreamForRiver(stream_factory())

    riverResults = _evaluateStreamOnRiver(riverStream, lr, b_lr, l1, l2, clip, bias_init)

    table = []
    for key in ["Accuracy", "F1", "Precision", "Recall"]:
        capy = capyMoaResults[key]
        river = riverResults[key]

        table.append([
            key,
            f"{capy:.2f}%",
            f"{river:.2f}%",
            f"{(capy - river):+.2f}%"
        ])

    print("\n--- Comparison CapyMOA vs River ---\n")
    print(
        tabulate(
            table,
            headers=["Metric", "CapyMOA", "River", "Delta"],
            tablefmt="fancy_grid"
        )
    )

def _evaluateStreamOnCapyMoa(stream, lr, b_lr, l1, l2, clip, bias_init):
    log_reg_capymoa = LogisticRegression(
        schema=stream.get_schema(),
        learning_rate=lr,
        bias_learning_rate=b_lr,
        l1_penalty=l1,
        l2_penalty=l2,
        clip_gradient=clip,
        bias_init=bias_init
    )

    y_true = []
    y_pred = []

    # prequential evaluation
    for i, instance in enumerate(stream):
        if i >= MAX_INSTANCES:
            break

        # test
        votes = log_reg_capymoa.predict_proba(instance)

        if votes is None or len(votes) == 0:
            pred = False
        else:
            pred = max(range(len(votes)), key=lambda j: votes[j])

        true_label = int(instance.y_index)

        y_true.append(true_label)
        y_pred.append(pred)

        # train
        log_reg_capymoa.train(instance)

    return _compute_sklearn_metrics(y_true, y_pred)

def _evaluateStreamOnRiver(stream, lr, b_lr, l1, l2, clip, bias_init):
    log_reg_river = linear_model.LogisticRegression(
        optimizer=optim.SGD(lr),
        intercept_lr=b_lr,
        l1=l1,
        l2=l2,
        clip_gradient=clip,
        intercept_init=bias_init
    )

    y_true = []
    y_pred = []

    # prequential evaluation
    for i, (x, y) in enumerate(stream):
        if i >= MAX_INSTANCES:
            break

        pred = log_reg_river.predict_one(x)

        if pred is None:
            pred = False

        y_true.append(y)
        y_pred.append(pred)

        log_reg_river.learn_one(x, y)

    return _compute_sklearn_metrics(y_true, y_pred)

def _compute_sklearn_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred) * 100,
        "F1": f1_score(y_true, y_pred, zero_division=0) * 100,
        "Precision": precision_score(y_true, y_pred, zero_division=0) * 100,
        "Recall": recall_score(y_true, y_pred, zero_division=0) * 100,
    }

## Electricity dataset

In [ ]:
from capymoa.datasets import Electricity

evaluateStream(Electricity)

## ElectricityTiny dataset

In [ ]:
from capymoa.datasets import ElectricityTiny

evaluateStream(ElectricityTiny)

## RandomRBFGenerator

In [ ]:
from capymoa.stream.generator import RandomRBFGenerator

def make_stream():
    return RandomRBFGenerator(
        number_of_classes=2,
        number_of_attributes=50,
        number_of_centroids=100,
        model_random_seed=SEED
    )

evaluateStream(make_stream)

## Hyper100k dataset

In [ ]:
from capymoa.datasets import Hyper100k

evaluateStream(Hyper100k)

## SEA dataset generator

In [ ]:
from capymoa.stream.generator import SEA

def make_stream():
    return SEA(
        instance_random_seed=SEED,
        function=1,
        balance_classes=False,
        noise_percentage=10,
    )

evaluateStream(make_stream)

## HyperPlaneClassification dataset

In [ ]:
from capymoa.stream.generator import HyperPlaneClassification

def make_stream():
    return HyperPlaneClassification(
        instance_random_seed=SEED,
        number_of_classes=2,
        number_of_attributes=10,
        number_of_drifting_attributes=2,
        magnitude_of_change=0.0,
        noise_percentage=5,
        sigma_percentage=10,
    )

evaluateStream(make_stream)

## RandomTreeGenerator

In [ ]:
from capymoa.stream.generator import RandomTreeGenerator

def make_stream():
    return RandomTreeGenerator(
        instance_random_seed=SEED,
        tree_random_seed=SEED,
        num_classes=2,
        num_nominals=0,
        num_numerics=5,
        max_tree_depth=5,
        first_leaf_level=3,
        leaf_fraction=0.15,
    )

evaluateStream(make_stream)

## DriftStream dataset generator
(reference: 04_drift_streams capymoa's tutorial)

In [ ]:
from capymoa.stream.drift import DriftStream, AbruptDrift, GradualDrift
from capymoa.stream.generator import SEA

def make_stream():
    return DriftStream(
        stream=[
            SEA(function=1), # first concept
            AbruptDrift(position=5000),
            SEA(function=3), # second concept
            GradualDrift(position=10000, width=3000),
            SEA(function=1), # third concept
        ]
    )

evaluateStream(make_stream)

## Electricity dataset (changed model parameters)

### L2

In [ ]:
from capymoa.datasets import Electricity

evaluateStream(Electricity, l2=0.01)

### L1

In [ ]:
from capymoa.datasets import Electricity

evaluateStream(Electricity, l1=0.01)

### Learning rate

In [ ]:
from capymoa.datasets import Electricity

evaluateStream(Electricity, lr=0.1, b_lr=0.1)

### Gradient clipping

In [ ]:
from capymoa.datasets import Electricity

evaluateStream(Electricity, clip=1)

### Bias initialization

In [ ]:
from capymoa.datasets import Electricity

evaluateStream(Electricity, bias_init=3)